<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); padding: 40px 32px 32px 32px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: #ffffff; font-size: 2.0em; margin: 0 0 6px 0; font-family: 'Segoe UI', sans-serif; font-weight: 700;">
    Capstone Project -- Porosity Prediction from Well Logs
  </h1>
  <h2 style="color: #a8d4f5; font-size: 1.2em; margin: 0 0 18px 0; font-family: 'Segoe UI', sans-serif; font-weight: 400;">
    Zedela Oluoch &middot; Track A &middot; Can we accurately predict neutron porosity in an unseen well using standard wireline logs?
  </h2>
  <hr style="border: 1px solid rgba(255,255,255,0.25); margin: 16px 0;">
  <table style="color: #cce4ff; font-family: 'Segoe UI', sans-serif; font-size: 0.95em;">
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Week:</strong></td><td>15-16 of 16 -- Capstone Project</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Module:</strong></td><td>Capstone: Designing and Delivering Your Own ML Project</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Prerequisites:</strong></td><td>Capstone Parts 1 and 2</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Course:</strong></td><td>Machine Learning for Petroleum Engineers &amp; Geoscientists</td>
    </tr>
  </table>
</div>

## 1. Project Charter

| Item | My answer |
|---|---|
| **Question** | Can machine learning accurately predict Neutron Porosity (NPHI) in a newly drilled well using only standard wireline logs? |
| **Who would use the answer, and for what decision?** | A production geologist or petrophysicist to estimate formation porosity where the NPHI tool was not run or experienced tool failure. |
| **Target variable (units)** | Neutron Porosity (`NPHI_frac`), measured in decimal fraction (v/v). |
| **Input features, and why each is physically sensible** | `GR_API` (indicates shale volume), `RHOB_gcc` (density responds directly to porosity), and `RT_ohm-m` (deep resistivity flags fluid changes). |
| **Features I will NOT use, and why** | `DEPTH`. Your instructor demonstrated that adding depth causes spatial data leakage, making the model score deceptively high on a random split but terrible on an unseen well. |
| **Evaluation split, and why it matches real use** | **Leave-One-Well-Out (Blind Well Split)**. One entire well will be completely hidden from training and used only for testing to mimic real operational deployment. |
| **Primary metric (units)** | Root Mean Squared Error (RMSE) in porosity units, alongside the Coefficient of Determination (R²). |
| **Baseline(s)** | 1. **Naïve Baseline:** Predicting the global median NPHI value of the training wells. <br>2. **Classical ML Baseline:** An untuned Linear Regression model. |
| **Success criterion** | Achieving a lower RMSE and a higher R² on the blind test well compared to both baselines. |
| **Biggest risk** | High geological variability between the training wells and the chosen blind test well, leading to poor generalization. |

In [1]:
## 2. Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = "../data/well_log_data.csv"
LOG_PATH = "../results/capstone_results_log.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

def log_result(track, model, split, metrics, notes=""):
    """Append one experiment (including failures) to the results log."""
    row = {"timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "track": track, "model": model,
           "split": split, **{k: round(float(v), 4) for k, v in metrics.items()}, "notes": notes}
    log = pd.read_csv(LOG_PATH) if os.path.exists(LOG_PATH) else pd.DataFrame()
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    print(f"Successfully logged experiment: {model}")
    return log

print("Setup completed successfully. Ready for data loading!")

Setup completed successfully. Ready for data loading!


In [11]:
import os
import pandas as pd

# Using the DATA_DIR variable from your setup section
csv_path = os.path.join(DATA_DIR, "well_log_data.csv")
df_check = pd.read_csv(csv_path, nrows=5)
print(df_check.columns.tolist())

NameError: name 'DATA_DIR' is not defined

In [12]:
import pandas as pd
df_check = pd.read_csv("../data/well_log_data.csv", nrows=5)
print(df_check.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '../data/well_log_data.csv'